# 04 · Índice de rotación y generadores de tráfico

**Objetivo:** construir, por hexágono, (1) qué tanto se movieron los restaurantes entre 2020 y 2025 y (2) qué atrae gente a la zona además de sus habitantes. El resultado es la tabla del modelo explicativo (notebook 05).

**Índice de rotación**

$$\text{rotación} = \frac{\text{salidas} + \text{entradas}}{\text{restaurantes en 2020}}$$

Un valor de 1 quiere decir que se movieron tantos restaurantes como los que había al inicio. Se reportan también sus partes: **permanencia** (siguen / había), **tasa de salida** y **tasa de entrada**.

**Cautela:** las entradas están infladas, sobre todo en la periferia, porque el Censo Económico 2024 incorporó negocios que ya existían pero no estaban en el directorio. La permanencia no tiene ese problema; por eso el notebook 05 la usa como resultado principal.

**Generadores de tráfico:** establecimientos que atraen personas durante el día (escuelas, servicios de salud, oficinas, gobierno, comercio grande, hoteles y entretenimiento) y el empleo estimado. Se miden en **2020**, antes del cambio, para que expliquen lo que pasó después y no al revés.

## 0. Librerías y rutas

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h3
import mapas

ROOT = Path.cwd().parent
PROC, FIG = ROOT / "data/processed", ROOT / "figures"

## 1. Configuración

- `MIN_BASE`: mínimo de restaurantes en 2020 para calcular tasas por hexágono. Con menos, una sola salida mueve la tasa decenas de puntos.
- `GENERADORES`: prefijos SCIAN de cada generador de tráfico. El gobierno (SCIAN 93) se lee de su propio archivo.
- `EMPLEO_PUNTO_MEDIO`: el DENUE da el personal ocupado en rangos; se usa el punto medio de cada rango para estimar el empleo.

In [ ]:
BASE, FIN, RES = "2020-04", "2025-05", 8
MIN_BASE = 5

GENERADORES = {
    "escuelas": ("611",),                          # servicios educativos
    "salud": ("621", "622"),                       # consultorios, clínicas y hospitales
    "oficinas": ("52", "54", "55", "56"),          # financieros, profesionales, corporativos y de apoyo
    "comercio_grande": ("4621", "4622"),           # tiendas de autoservicio y departamentales
    "ocio": ("71", "7211"),                        # entretenimiento y hoteles
}
EMPLEO_PUNTO_MEDIO = {"0 a 5 personas": 3, "6 a 10 personas": 8, "11 a 30 personas": 20, "31 a 50 personas": 40,
                      "51 a 100 personas": 75, "101 a 250 personas": 175, "251 y más personas": 300}

## 2. Rotación por hexágono

In [ ]:
matriz = pd.read_parquet(PROC / f"matriz_hex{RES}_tipo_2020_2025.parquet")
trans = pd.read_parquet(PROC / "transiciones_restaurantes_2020_2025.parquet")
ent = pd.read_parquet(PROC / "entradas_restaurantes_2020_2025.parquet")

def tasas(df):
    df = df.copy()
    df["permanencia"] = df["continuan"] / df["n_base"]
    df["tasa_salida"] = df["salidas"] / df["n_base"]
    df["tasa_entrada"] = df["entradas"] / df["n_base"]
    df["rotacion"] = (df["salidas"] + df["entradas"]) / df["n_base"]
    df["neto"] = df["entradas"] - df["salidas"]
    return df

rot_hex = tasas(matriz.groupby("h3")[["n_base", "continuan", "salidas", "entradas"]].sum().query("n_base > 0"))
estables = rot_hex[rot_hex["n_base"] >= MIN_BASE]
print(f"Hexágonos con restaurantes en 2020: {len(rot_hex):,} · con al menos {MIN_BASE}: {len(estables):,}")
print(f"Rotación de la ZM: {(trans['sale'].sum() + len(ent)) / len(trans):.2f}")
estables[["permanencia", "tasa_salida", "tasa_entrada", "rotacion"]].describe().round(2)

## 3. Rotación por tipo de restaurante y por municipio

Tasas calculadas sobre todos los restaurantes de cada grupo (no promedio de hexágonos).

In [ ]:
def resumen_por(col):
    g = trans.groupby(col).agg(n_base=("id", "size"), continuan=("continua", "sum"), salidas=("sale", "sum"))
    g["entradas"] = ent.groupby(col).size()
    return tasas(g.fillna(0))[["n_base", "permanencia", "tasa_salida", "tasa_entrada", "rotacion", "neto"]]

por_tipo = resumen_por("tipo").sort_values("rotacion", ascending=False)
por_tipo.round(2)

In [ ]:
resumen_por("mun_nombre").sort_values("rotacion", ascending=False).round(2)

## 4. Rotación por tipo de zona

El tipo de zona viene del notebook 02 (población y restaurantes de 2020 contra la mediana). Los hexágonos que no están en la tabla base (sin población ni establecimientos en 2020) quedan como *Nuevos*.

In [ ]:
base_hex = pd.read_parquet(PROC / f"hex_res{RES}_base_2020.parquet").set_index("h3")
todos = matriz.groupby("h3")[["n_base", "continuan", "salidas", "entradas"]].sum()
todos["tipo_zona"] = base_hex["tipo_zona"].reindex(todos.index).fillna("Nuevos (sin datos en 2020)")
por_zona = tasas(todos.groupby("tipo_zona")[["n_base", "continuan", "salidas", "entradas"]].sum().query("n_base > 0"))
por_zona["hexagonos"] = todos.groupby("tipo_zona").size()
nuevos = todos[(todos["n_base"] == 0) & (todos["entradas"] > 0)]
print(f"Hexágonos sin restaurantes en 2020 que tienen entradas en 2025: {len(nuevos):,} ({nuevos['entradas'].sum():,} restaurantes)")
por_zona[["hexagonos", "n_base", "permanencia", "tasa_entrada", "rotacion", "neto"]].round(2)

## 5. Mapa · Índice de rotación por hexágono

In [ ]:
BORDES = [0, 0.5, 0.8, 1.1, 1.5, 2.0, max(2.1, estables["rotacion"].max() + 0.01)]
ETIQ = ["< 0.5", "0.5–0.8", "0.8–1.1", "1.1–1.5", "1.5–2.0", "≥ 2.0"]
cmap, norm = mapas.escala_por_clases(BORDES, ["#EAF1F6", "#BFD6E6", "#8DB7D3", "#E8A25F", "#C2410C", "#7F2704"])
fig, ax, polys = mapas.base_mapa(estables.index)
coll = mapas.capa_hex(ax, polys, array=estables["rotacion"].values, cmap=cmap, norm=norm)
mapas.referencias(ax)
mapas.barra_clases(fig, ax, coll, BORDES, ETIQ, "(salidas + entradas) / restaurantes 2020")
ax.set_title(f"Índice de rotación 2020 → 2025 · hexágonos con ≥ {MIN_BASE} restaurantes", loc="left", fontsize=13)
plt.tight_layout(); plt.savefig(FIG / "04_mapa_rotacion.png", dpi=200, bbox_inches="tight"); plt.show()

## 6. Generadores de tráfico por hexágono

Para cada generador se cuentan los establecimientos:

- **en el hexágono**, y
- **en su entorno**: el hexágono más sus 6 vecinos (alrededor de 5 km²), porque la gente camina o maneja unas cuadras para comer.

El empleo estimado suma el punto medio del personal ocupado de todos los establecimientos, incluido el gobierno. Se calcula para 2020 (el que entra al modelo) y para 2025 (para describir el cambio).

In [ ]:
def a_hex(df):
    return df.assign(h3=[h3.latlng_to_cell(la, lo, RES) for la, lo in zip(df["latitud"], df["longitud"])])

def generadores(edicion):
    den = a_hex(pd.read_parquet(PROC / f"denue_zmq_{edicion}.parquet"))
    gob = a_hex(pd.read_parquet(PROC / f"gobierno_zmq_{edicion}.parquet"))
    todos_est = pd.concat([den, gob])
    g = pd.DataFrame(index=pd.Index(sorted(todos_est["h3"].unique()), name="h3"))   # todos los hexágonos con negocios
    cod = den["codigo_act"].astype(str)
    for nombre, pref in GENERADORES.items():
        g[nombre] = den[cod.str.startswith(pref)].groupby("h3").size()
    g["gobierno"] = gob.groupby("h3").size()
    g["empleo"] = todos_est["per_ocu"].map(EMPLEO_PUNTO_MEDIO).groupby(todos_est["h3"]).sum()
    g["establecimientos"] = todos_est.groupby("h3").size()
    g["municipio"] = todos_est.groupby("h3")["mun_nombre"].agg(lambda s: s.mode().iat[0])
    num = g.columns.drop("municipio")
    g[num] = g[num].fillna(0)
    return g

def entorno(g, cols):
    # Suma de cada columna en el hexágono y sus 6 vecinos
    out = {}
    for c in g.index:
        vecinos = [v for v in h3.grid_disk(c, 1) if v in g.index]
        out[c] = g.loc[vecinos, cols].sum()
    return pd.DataFrame(out).T.add_suffix("_entorno")

gen20, gen25 = generadores(BASE), generadores(FIN)
COLS_GEN = list(GENERADORES) + ["gobierno", "empleo"]
gen20 = gen20.join(entorno(gen20, COLS_GEN + ["establecimientos"]))
cambio_gen = pd.DataFrame({"2020": gen20[COLS_GEN].sum(), "2025": gen25[COLS_GEN].sum()})
cambio_gen["cambio %"] = ((cambio_gen["2025"] / cambio_gen["2020"] - 1) * 100).round(1)
cambio_gen

## 7. Tabla del modelo

In [ ]:
# Competencia: restaurantes de 2020 en el hexágono y su entorno
rest20 = rot_hex["n_base"].reindex(gen20.index).fillna(0).rename("restaurantes")
comp = entorno(rest20.to_frame(), ["restaurantes"])

modelo = (rot_hex
          .join(base_hex[["POBTOT", "TVIVHAB", "POCUPADA", "VPH_AUTOM", "escolaridad", "tipo_zona"]], how="left")
          .join(gen20, how="left")
          .join(comp, how="left"))
modelo["autos_por_vivienda"] = modelo["VPH_AUTOM"] / modelo["TVIVHAB"].replace(0, np.nan)
modelo["tipo_zona"] = modelo["tipo_zona"].fillna("Nuevos (sin datos en 2020)")
modelo = modelo.rename_axis("h3").reset_index()
modelo.to_parquet(PROC / f"hex_res{RES}_modelo.parquet", index=False)
print(f"Tabla del modelo: {len(modelo):,} hexágonos × {modelo.shape[1]} columnas → hex_res{RES}_modelo.parquet")
modelo.head()

## 8. Primer vistazo: ¿con qué se relacionan la permanencia y la rotación?

Correlación de rangos (Spearman) entre hexágonos con al menos `MIN_BASE` restaurantes. Es descriptiva: el notebook 05 controla por tipo de restaurante, municipio y competencia al mismo tiempo.

In [ ]:
VARS = {"POBTOT": "Población", "escolaridad": "Escolaridad", "autos_por_vivienda": "Autos por vivienda",
        "restaurantes_entorno": "Restaurantes en el entorno", "empleo_entorno": "Empleo en el entorno",
        "escuelas_entorno": "Escuelas", "salud_entorno": "Salud", "oficinas_entorno": "Oficinas",
        "gobierno_entorno": "Gobierno", "comercio_grande_entorno": "Comercio grande", "ocio_entorno": "Ocio y hoteles"}
k = modelo[modelo["n_base"] >= MIN_BASE]
corr = pd.DataFrame({res: [k[v].corr(k[res], method="spearman") for v in VARS] for res in ("permanencia", "rotacion")},
                    index=list(VARS.values()))

fig, axes = plt.subplots(1, 2, figsize=(12, 6), sharey=True)
for ax, res, color in zip(axes, ("permanencia", "rotacion"), ("#14213D", "#A8551A")):
    serie = corr[res].iloc[::-1]
    ax.barh(serie.index, serie.values, color=color)
    ax.axvline(0, color="#999", lw=1)
    for y, v in enumerate(serie.values):
        ax.text(v + (0.01 if v >= 0 else -0.01), y, f"{v:+.2f}", va="center", ha="left" if v >= 0 else "right", fontsize=9)
    ax.set_xlim(-0.5, 0.5); ax.set_title({"permanencia": "Permanencia", "rotacion": "Rotación"}[res], loc="left")
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle(f"Correlación de rangos con la permanencia y la rotación (hexágonos con ≥ {MIN_BASE} restaurantes, n = {len(k)})",
             x=0.01, ha="left")
plt.tight_layout(); plt.savefig(FIG / "04_correlaciones.png", dpi=200, bbox_inches="tight"); plt.show()
corr.round(2)